# 120 — Structured Phenotyping (Step 2)

**Dataset:** Structured Retrospective (task-planning app)
**Stage:** Step 2 — phenotyping across all 7 completed Step 1 model families (NeT/InT/StoP/DyS/SyD/StM/GaT)
**Blueprint:** `outputs/reports/structured/structured_phenotyping_blueprint.md` — read this first.
**Numbering:** 120 mirrors DS5's own jump from model notebooks to Step 2 at 100, shifted +20 to match
the Structured model decade's own +20 offset (60-68 = DS5's 41-48 +20).
**Output root:** `outputs/model_outputs/structured/phenotyping/`

---

## Scope (explicit, per user instruction to focus effort)

**One well-executed clustering pass, not DS5's six-path exploration.** Justified by sample size: DS5's
Phase 1 population was N=414; Structured's cross-model complete-case population is N=36 (checked in the
blueprint before writing any code). Six underpowered passes would be worse than one adequately-scoped one.

The radar chart (Part 4 of the blueprint) is the priority deliverable — reusing the exact function from
`notebooks/100_phenotype_exploration.ipynb` cell 26, adapted for Structured's 7 model families.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import OrderedDict
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "structured"
FEATURES = PROJECT_ROOT / "outputs" / "features" / "structured"
OUT = MO / "phenotyping"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Output folder:", OUT)

## 1. Load All 7 Model Outputs and Build the Feature Matrix (Part 1 of blueprint)

In [ ]:
net = pd.read_csv(MO / "network_theory" / "participant_network_metrics.csv")
intt = pd.read_csv(MO / "information_theory" / "participant_entropy_metrics.csv")
stop = pd.read_csv(MO / "stochastic_processes" / "participant_stop_parameters.csv")
dys = pd.read_csv(MO / "dynamical_systems" / "participant_trajectory_metrics.csv")
syd = pd.read_csv(MO / "system_dynamics" / "participant_syd_summary.csv")
stm = pd.read_csv(MO / "statistical_mechanics" / "participant_stm_summary.csv")
gat = pd.read_csv(MO / "game_theory" / "participant_gat_summary.csv")

# One curated, non-redundant feature per model family (see blueprint Part 1 for the redundancy checks)
FEATURE_MAP = OrderedDict([
    ("HHI_net", ("NeT", net[["participant_id", "HHI_net"]])),
    ("H_completion", ("InT", intt[["participant_id", "H_completion"]])),
    ("created_Fano_factor", ("StoP", stop[["participant_id", "created_Fano_factor"]])),
    ("state_space_volatility", ("DyS", dys[["participant_id", "state_space_volatility"]])),
    ("backlog_trend_slope", ("SyD", syd[["participant_id", "backlog_trend_slope"]])),
    ("susceptibility", ("StM", stm[["participant_id", "susceptibility"]])),
    ("mean_commitment_score", ("GaT", gat[["participant_id", "mean_commitment_score"]])),
])

feat_df = None
for col, (family, df) in FEATURE_MAP.items():
    feat_df = df if feat_df is None else feat_df.merge(df, on="participant_id", how="outer")

feature_cols = list(FEATURE_MAP.keys())
complete_case = feat_df.dropna(subset=feature_cols).reset_index(drop=True)

print(f"All participants with any of the 7 features: {len(feat_df)}")
print(f"Complete-case (all 7 features present): {len(complete_case)}")
complete_case.head()

## 2. Feature Redundancy Report (Part 1/3 of blueprint)

In [ ]:
corr_matrix = complete_case[feature_cols].corr()
print(corr_matrix.round(3))

flagged = []
for i, c1 in enumerate(feature_cols):
    for c2 in feature_cols[i+1:]:
        r = corr_matrix.loc[c1, c2]
        if abs(r) > 0.6:
            flagged.append((c1, c2, round(r, 3)))

print()
if flagged:
    print("Pairs with |r| > 0.6 (unexpected -- would need addressing):")
    for c1, c2, r in flagged:
        print(f"  {c1} <-> {c2}: r={r}")
else:
    print("No unexpected redundant pairs (|r| > 0.6) among the 7 curated features.")

corr_matrix.to_csv(OUT / "feature_redundancy_report.csv")

## 3. Save Feature Matrix

In [ ]:
complete_case.to_csv(OUT / "phenotype_feature_matrix.csv", index=False)
print(f"Saved: phenotype_feature_matrix.csv  ({complete_case.shape})")

## 4. Figure — Feature Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Structured phenotyping -- feature correlation matrix (N={})".format(len(complete_case)))
plt.tight_layout()
plt.savefig(FIG / "fig3_feature_correlation_matrix.png", dpi=150)
plt.show()
print("Saved fig3_feature_correlation_matrix.png")

## 5. K-Selection (Part 3 of blueprint)

`k=2..4`, minimum cell size >= 10 (N-adjusted from DS5's >=30, proportionate to N=36 vs. DS5's 414).

In [ ]:
X = complete_case[feature_cols].to_numpy()
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_z = (X - X_mean) / X_std

MIN_CELL = 10
k_results = {}
for k in range(2, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_z)
    labels = km.labels_
    sizes = pd.Series(labels).value_counts()
    sil = silhouette_score(X_z, labels)
    ch = calinski_harabasz_score(X_z, labels)
    db = davies_bouldin_score(X_z, labels)
    k_results[k] = {"labels": labels, "silhouette": sil, "CH": ch, "DB": db,
                     "min_size": int(sizes.min()), "sizes": sizes.to_dict()}
    print(f"k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, "
          f"min_cell={sizes.min()}, sizes={dict(sizes)}")

valid_ks = [k for k, r in k_results.items() if r["min_size"] >= MIN_CELL]
print(f"\nValid k (min_cell >= {MIN_CELL}): {valid_ks}")
SELECTED_K = max(valid_ks, key=lambda k: k_results[k]["silhouette"]) if valid_ks else 2
print(f"Selected k: {SELECTED_K}")

## 6. Figure — K-Selection Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(k_results.keys())
axes[0].plot(ks, [k_results[k]["silhouette"] for k in ks], marker="o")
axes[0].set_title("Silhouette (higher better)")
axes[0].set_xlabel("k")
axes[1].plot(ks, [k_results[k]["CH"] for k in ks], marker="o", color="#55A868")
axes[1].set_title("Calinski-Harabasz (higher better)")
axes[1].set_xlabel("k")
axes[2].plot(ks, [k_results[k]["DB"] for k in ks], marker="o", color="#C44E52")
axes[2].set_title("Davies-Bouldin (lower better)")
axes[2].set_xlabel("k")
for ax in axes:
    ax.axvline(SELECTED_K, color="gray", ls="--", lw=1)
plt.tight_layout()
plt.savefig(FIG / "fig1_k_selection_metrics.png", dpi=150)
plt.show()
print("Saved fig1_k_selection_metrics.png")

## 7. Bootstrap Jaccard Stability Check

In [ ]:
def bootstrap_jaccard(X_z, labels_full, k, n_boot=100, frac=0.8, seed=RANDOM_STATE):
    n = len(X_z)
    rng = np.random.RandomState(seed)
    jaccards = []
    for b in range(n_boot):
        idx = rng.choice(n, size=int(n * frac), replace=False)
        km_sub = KMeans(n_clusters=k, n_init=10, random_state=seed + b).fit(X_z[idx])
        labels_sub = km_sub.labels_
        labels_full_sub = labels_full[idx]

        cont = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                cont[i, j] = np.sum((labels_full_sub == i) & (labels_sub == j))
        row_ind, col_ind = linear_sum_assignment(-cont)

        pair_jaccards = []
        for i, j in zip(row_ind, col_ind):
            set_full = set(np.where(labels_full_sub == i)[0])
            set_sub = set(np.where(labels_sub == j)[0])
            union = len(set_full | set_sub)
            pair_jaccards.append(len(set_full & set_sub) / union if union > 0 else np.nan)
        jaccards.append(np.nanmean(pair_jaccards))
    return float(np.mean(jaccards)), float(np.std(jaccards))

jac_mean, jac_std = bootstrap_jaccard(X_z, k_results[SELECTED_K]["labels"], SELECTED_K)
stability_verdict = "STABLE" if jac_mean >= 0.75 else "MARGINAL" if jac_mean >= 0.60 else "UNSTABLE"
print(f"Bootstrap Jaccard (k={SELECTED_K}, 100 resamples, 80% subsample): "
      f"mean={jac_mean:.3f}, std={jac_std:.3f}")
print(f"Stability verdict: {stability_verdict}")
print("(N=36 caveat: this statistic is noisier here than in DS5's N=414 analysis -- interpret accordingly)")

## 8. PCA Sanity Check

In [ ]:
pca = PCA(n_components=len(feature_cols))
pca.fit(X_z)
print("Variance explained per component:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v*100:.1f}%")
print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 6))
coords = pca.transform(X_z)[:, :2]
labels_sel = k_results[SELECTED_K]["labels"]
for lab in np.unique(labels_sel):
    mask = labels_sel == lab
    ax.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {lab} (n={mask.sum()})", alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"PCA biplot, k={SELECTED_K}")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_pca_biplot.png", dpi=150)
plt.show()
print("Saved fig2_pca_biplot.png")

## 9. Cluster Assignments and Profiles

In [ ]:
PROFILE_NAMES = [f"Profile {chr(65+i)}" for i in range(SELECTED_K)]  # Profile A, B, C...
label_to_name = {lab: PROFILE_NAMES[i] for i, lab in enumerate(sorted(np.unique(labels_sel)))}

complete_case["cluster_label"] = [label_to_name[l] for l in labels_sel]

cluster_assignments = complete_case[["participant_id", "cluster_label"] + feature_cols].copy()
cluster_assignments.to_csv(OUT / "cluster_assignments.csv", index=False)

cluster_profiles = complete_case.groupby("cluster_label")[feature_cols].agg(["mean", "median", "std"])
cluster_profiles.to_csv(OUT / "cluster_profiles.csv")

print(f"Saved: cluster_assignments.csv ({cluster_assignments.shape}), cluster_profiles.csv")
print()
print("Cluster sizes:")
print(complete_case["cluster_label"].value_counts())
print()
print("Cluster means (raw, unstandardized):")
print(complete_case.groupby("cluster_label")[feature_cols].mean().round(4))

## 10. The Enhanced Radar Chart (Part 4 of blueprint — priority deliverable)

Adapted from `notebooks/100_phenotype_exploration.ipynb` cell 26 (`plot_phenotype_profiles_enhanced`).
Same visual design (individual spaghetti lines, IQR band, bold mean line with white-centered markers,
boxed axis labels, dual concentric arc/band rings, dark polar grid, no title). Adapted: 7 axes now map
1:1 onto the 7 model families (cleaner than the DS5 original, which doubled up some families);
conceptual-layer bands grounded in the theory paper's own Figure 1 grouping (StM/StoP/DyS/SyD vs.
NeT/InT/GaT), split into 3 bands for visual balance (State & Process / Feedback & Flow /
Structural & Strategic) as documented in the blueprint.

In [ ]:
# Font sizes matched to notebooks/100_phenotype_exploration.ipynb cell 26 (the
# source of the DS5 reference figure, Figure 4.12) -- see
# scripts/regenerate_structured_phenotype_radar.py, which is the actual script
# that produced the current fig4_phenotype_profiles_enhanced_notitle.png (loads
# the frozen cluster_assignments.csv rather than re-running the clustering above).
FONT_SIZES = {
    "axis_labels": 18, "section_layers": 18, "grid_ticks": 15,
    "bottom_legends": 18, "legend_titles": 18,
}

RADAR_DIMS = OrderedDict([
    ("Structure\nsusceptibility", "susceptibility"),            # StM
    ("Burstiness",                 "created_Fano_factor"),        # StoP
    ("Trajectory\ninstability",   "state_space_volatility"),     # DyS
    ("Backlog\ntrend",            "backlog_trend_slope"),        # SyD
    ("Network\nconcentration",    "HHI_net"),                    # NeT
    ("Completion\npredictability","H_completion"),                # InT
    ("Mean\ncommitment",          "mean_commitment_score"),       # GaT
])

AXIS_MODEL = {
    "Structure\nsusceptibility": "StM", "Burstiness": "StoP",
    "Trajectory\ninstability": "DyS", "Backlog\ntrend": "SyD",
    "Network\nconcentration": "NeT", "Completion\npredictability": "InT",
    "Mean\ncommitment": "GaT",
}

MODEL_COLORS = {
    "NeT": "#4CAF50", "InT": "#2196F3", "StoP": "#FF9800",
    "DyS": "#3F51B5", "SyD": "#00BCD4", "StM": "#9C27B0", "GaT": "#F44336",
}

PROFILE_COLOR_CYCLE = ["#A20A7C", "#7E57C2", "#00897B", "#EF6C00"]
PROFILE_COLORS = {name: PROFILE_COLOR_CYCLE[i] for i, name in enumerate(PROFILE_NAMES)}

# 2-2-3 grouping over the RADAR_DIMS order above (indices 0-6)
SECTION_LAYERS = [
    {"name": "State & Process",        "dim_idx": [0, 1],    "color": "#C62828"},
    {"name": "Feedback & Flow",        "dim_idx": [2, 3],    "color": "#0A4B96"},
    {"name": "Structural & Strategic", "dim_idx": [4, 5, 6], "color": "#2E7D32"},
]

print("Radar configuration set for", len(RADAR_DIMS), "axes,", SELECTED_K, "profiles.")

In [ ]:
def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors, arc_r=1.04, arc_lw=3.5, alpha=0.45):
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.40
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, "")
        color = model_colors.get(model, "#AAAAAA")
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r), color=color, lw=arc_lw,
                solid_capstyle="round", alpha=alpha, clip_on=False, zorder=10)


def draw_section_layers(ax, angles, section_layers, band_inner=1.09, band_outer=1.18, label_r=1.135):
    step = angles[1] - angles[0]
    half = step / 2
    for sec in section_layers:
        idx = sec["dim_idx"]
        color = sec["color"]
        a0 = angles[idx[0]] - half
        a1 = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)
        ax.fill_between(theta, band_inner, band_outer, color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner), color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer), color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)


def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.30):
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label, ha="center", va="center",
                fontsize=FONT_SIZES["axis_labels"], color="#222222",
                clip_on=False, zorder=25, multialignment="center",
                bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="#BBBBBB", lw=0.9, alpha=0.97))


def compute_iqr_band(sub_df, norm_cols, angles):
    q25 = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75 = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name, color, dim_labels,
                         axis_model, model_colors, section_layers,
                         individual_alpha=0.25, individual_lw=0.7, mean_lw=2.8):
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls="--", zorder=4)

    ax.plot(angles, mean_vals, "o-", color=color, lw=mean_lw, ms=5,
            markerfacecolor="white", markeredgecolor=color, markeredgewidth=1.8,
            label=f"{profile_name} mean  (n={len(sub_df)})", zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([""] * len(dim_labels))
    ax.tick_params(axis="x", length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=FONT_SIZES["grid_ticks"],
                        alpha=0.5, color="#555555")
    ax.spines["polar"].set_color("#272727")
    ax.grid(color="#121111", lw=0.6)


def plot_phenotype_profiles_enhanced(pop_df, profile_col, participant_col, profile_names,
                                       profile_colors, save_path=None):
    available = OrderedDict((k, v) for k, v in RADAR_DIMS.items() if v in pop_df.columns)
    dim_labels = list(available.keys())
    dim_feats = list(available.values())
    am = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(pop_df, dim_feats)
    radar_df = pop_df.copy()
    for feat in dim_feats:
        radar_df[f"{feat}_norm"] = norm_map[feat]
    norm_cols = [f"{feat}_norm" for feat in dim_feats]

    angles = np.linspace(0, 2 * np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    n_profiles = len(profile_names)
    fig, axes = plt.subplots(1, n_profiles, figsize=(9 * n_profiles, 13.5), subplot_kw=dict(polar=True))
    if n_profiles == 1:
        axes = [axes]
    # bottom=0.36 (up from 0.27) and legend fontsize 18 (up from 13). 7 model
    # families now stay in 1 column (was 2, to fit at the old smaller size), so
    # the legend moved from a lower-center split to lower-left/lower-right,
    # mirroring the DS5/DS6 layout. Verified visually against overlap with the
    # outer panels' axis labels; see scripts/regenerate_structured_phenotype_radar.py.
    fig.subplots_adjust(wspace=0.6, left=0.06, right=0.94, top=0.80, bottom=0.36)

    for pname, ax in zip(profile_names, axes):
        color = profile_colors[pname]
        sub = radar_df[radar_df[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f"{pname}\n(no data)")
            continue
        plot_single_profile(ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
                             profile_name=pname, color=color, dim_labels=dim_labels,
                             axis_model=am, model_colors=MODEL_COLORS, section_layers=SECTION_LAYERS)

    def _desc(m):
        return {"InT": "Information Theory", "StoP": "Stochastic Processes", "StM": "Statistical Mechanics",
                "NeT": "Network Theory", "DyS": "Dynamical Systems", "SyD": "System Dynamics",
                "GaT": "Game Theory"}.get(m, m)

    graph_patches = [mpatches.Patch(facecolor=profile_colors[p], label=f"{p} mean") for p in profile_names]
    graph_patches.append(mpatches.Patch(facecolor="grey", alpha=0.35, label="IQR band (25th-75th pct.)"))

    arc_patches = [mpatches.Patch(color=c, label=f"{m} — {_desc(m)}") for m, c in MODEL_COLORS.items()]
    sec_patches = [mpatches.Patch(color=s["color"], alpha=0.55, label=s["name"]) for s in SECTION_LAYERS]

    leg0 = fig.legend(handles=graph_patches, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=1,
                       fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
                       title="Profiles  (mean line + IQR)", title_fontsize=FONT_SIZES["legend_titles"])
    fig.add_artist(leg0)
    leg1 = fig.legend(handles=arc_patches, loc="lower left", bbox_to_anchor=(0.02, 0.02), ncol=1,
                       fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
                       title="Model families  (axis color)", title_fontsize=FONT_SIZES["legend_titles"])
    fig.add_artist(leg1)
    fig.legend(handles=sec_patches, loc="lower right", bbox_to_anchor=(0.98, 0.02), ncol=1,
               fontsize=FONT_SIZES["bottom_legends"], framealpha=0.95, edgecolor="#CCCCCC",
               title="Conceptual layers  (band color)", title_fontsize=FONT_SIZES["legend_titles"])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight", dpi=300)
        print(f"Saved: {save_path}")
    plt.show()
    return fig, axes

fig, axes = plot_phenotype_profiles_enhanced(
    pop_df=complete_case,
    profile_col="cluster_label",
    participant_col="participant_id",
    profile_names=PROFILE_NAMES,
    profile_colors=PROFILE_COLORS,
    save_path=FIG / "fig4_phenotype_profiles_enhanced_notitle.png",
)

## 11. Psychosocial Overlay (Part 5 of blueprint — secondary, exploratory)

In [ ]:
surv = pd.read_csv(FEATURES / "structured_survey_features.csv")
surv_usable = surv[surv["consent_given"] & surv["is_finished"]]

overlay_df = complete_case.merge(surv_usable, on="participant_id", how="inner")
print(f"N with cluster assignment AND usable survey data: {len(overlay_df)}")

SCALES = ["life_happiness_score", "productivity_happiness_score",
          "productivity_importance_general_score", "pct_tasks_planned_num",
          "pct_planned_tasks_completed_num"]
SCALES = [s for s in SCALES if s in overlay_df.columns]

overlay_results = []
if len(overlay_df) >= 10 and overlay_df["cluster_label"].nunique() >= 2:
    for scale in SCALES:
        groups = [g[scale].dropna().values for _, g in overlay_df.groupby("cluster_label") if len(g[scale].dropna()) >= 3]
        if len(groups) < 2:
            continue
        h_stat, p_val = stats.kruskal(*groups)
        overlay_results.append({"scale": scale, "H_stat": round(h_stat, 3), "p_raw": round(p_val, 4)})
    overlay_df_result = pd.DataFrame(overlay_results)
    if len(overlay_df_result) > 0:
        overlay_df_result["p_bonferroni"] = (overlay_df_result["p_raw"] * len(overlay_df_result)).clip(upper=1.0)
        print(overlay_df_result.to_string(index=False))
    else:
        print("Not enough non-null data per scale/cluster for a Kruskal-Wallis test.")
        overlay_df_result = pd.DataFrame(columns=["scale", "H_stat", "p_raw", "p_bonferroni"])
else:
    print(f"N too small ({len(overlay_df)}) or single-cluster -- skipping formal test, reporting as inconclusive.")
    overlay_df_result = pd.DataFrame(columns=["scale", "H_stat", "p_raw", "p_bonferroni"])

overlay_df_result.to_csv(OUT / "psychosocial_overlays.csv", index=False)
print(f"\nSaved: psychosocial_overlays.csv")

## 12. model_run_summary.md

In [ ]:
summary = f"""# Structured Phenotyping (Step 2) -- Model Run Summary

**Run date:** 2026-08-03
**Notebook:** notebooks/120_structured_phenotype_exploration.ipynb
**Blueprint:** outputs/reports/structured/structured_phenotyping_blueprint.md

## Sample
- Complete-case population (all 7 model families eligible): N={len(complete_case)}

## Feature redundancy
{"No unexpected redundant pairs (|r|>0.6)." if not flagged else str(flagged)}

## K-selection
"""
for k in k_results:
    r = k_results[k]
    summary += f"- k={k}: silhouette={r['silhouette']:.3f}, CH={r['CH']:.1f}, DB={r['DB']:.3f}, min_cell={r['min_size']}, sizes={r['sizes']}\n"
summary += f"""
**Selected k={SELECTED_K}** (silhouette={k_results[SELECTED_K]['silhouette']:.3f}, subject to min_cell>={MIN_CELL})

## Stability
- Bootstrap Jaccard (100 resamples, 80% subsample): mean={jac_mean:.3f}, std={jac_std:.3f}
- Verdict: {stability_verdict} (N=36 caveat: noisier than DS5's N=414 equivalent)

## PCA
- PC1+PC2 variance explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%

## Cluster profiles (raw means)
{complete_case.groupby("cluster_label")[feature_cols].mean().round(4).to_string()}

## Cluster sizes
{complete_case["cluster_label"].value_counts().to_string()}

## Psychosocial overlays (exploratory, N={len(overlay_df)})
{overlay_df_result.to_string(index=False) if len(overlay_df_result) > 0 else "Inconclusive -- N too small or single cluster."}

## Key limitations
1. N=36 complete-case population -- much smaller than DS5's N=414; all statistics noisier
2. One feature per model family chosen for interpretability; does not exhaust each model's full output
3. Radar individual-line density will look sparser than DS5's equivalent purely due to N
4. Psychosocial overlay N further thinned by consent/completion filtering -- exploratory only
5. k=2 ceiling expected and likely required by minimum cell size at this N
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)